# Module 03 — Storage I/O, Types & Iceberg (DSv2)

Compare formats and schemas, then a full Iceberg walkthrough: snapshots, time travel, partition evolution, and on-disk growth + compaction.

In [1]:
import sys
sys.path.insert(0, '/workspace')
from shared.spark_session import begin_module, ui_links, explain_plan
from shared import config
from shared.utils import timed_action, timed_read, drop_iceberg_table, reset_spark_configs
# from shared.demo_data import user_post_activity

spark = begin_module('module-03-storage-io')
spark.sparkContext.setLogLevel("ERROR")
ui_links()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/24 09:56:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


{'master_ui': 'http://localhost:8080',
 'app_ui': 'http://localhost:4040',
 'worker_1_ui': 'http://localhost:8081',
 'worker_2_ui': 'http://localhost:8082',
 'app_ui_internal': 'http://jupyter-lab:4040'}

In [2]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
trips = spark.read.parquet(*config.yellow_taxi_paths())
sample = trips.select(
    'VendorID', 'PULocationID', 'total_amount', 'tpep_pickup_datetime'
)
out = config.OUTPUT_BASE / '03-formats'
out.mkdir(parents=True, exist_ok=True)

In [3]:
ICEBERG = 'local.default.workshop_iceberg'
SHORT = 'workshop_iceberg'

## Write all formats

In [4]:
drop_iceberg_table(spark, 'local.default.io_lab_sample')

write_opts = {'maxRecordsPerFile': 500_000}

sample.write.mode('overwrite').partitionBy('VendorID').options(**write_opts).json(str(out / 'json'))
sample.write.mode('overwrite').partitionBy('VendorID').options(**write_opts).option('header', True).csv(str(out / 'csv'))
sample.write.mode('overwrite').partitionBy('VendorID').options(**write_opts).parquet(str(out / 'parquet'))

sample.writeTo('local.default.io_lab_sample').using('iceberg').partitionedBy('VendorID').create()

## Read benchmarks (count)

In [5]:
results = {}
_, results['json'] = timed_read(spark, lambda: spark.read.json(str(out / 'json')))
_, results['csv'] = timed_read(
    spark, lambda: spark.read.option('header', True).schema(sample.schema).csv(str(out / 'csv'))
)
_, results['parquet'] = timed_read(spark, lambda: spark.read.parquet(str(out / 'parquet')))
_, results['iceberg'] = timed_read(spark, lambda: spark.table('local.default.io_lab_sample'))
for k, t in sorted(results.items(), key=lambda x: -x[1]):
    print(f'{k:10s} {t:.2f}s')

read+count → 5,972,150 rows in 2.03s
read+count → 5,972,150 rows in 0.35s
read+count → 5,972,150 rows in 0.21s
read+count → 5,972,150 rows in 0.17s
json       2.03s
csv        0.35s
parquet    0.21s
iceberg    0.17s


## Type safety — CSV without schema

In [6]:
bad_csv = spark.read.option('header', True).csv(str(out / 'csv'))
bad_csv.printSchema()  # many strings
typed_csv = spark.read.schema(sample.schema).option('header', True).csv(str(out / 'csv'))
typed_csv.printSchema()

root
 |-- PULocationID: string (nullable = true)
 |-- total_amount: string (nullable = true)
 |-- tpep_pickup_datetime: string (nullable = true)
 |-- VendorID: integer (nullable = true)

root
 |-- PULocationID: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- VendorID: integer (nullable = true)



In [7]:
explain_plan(spark.read.schema(sample.schema).option('header', True).csv(str(out / 'csv')))

== Physical Plan ==
Scan csv  (1)


(1) Scan csv 
Output [4]: [PULocationID#108, total_amount#109, tpep_pickup_datetime#110, VendorID#111]
Batched: false
Location: InMemoryFileIndex [file:/workspace/data/output/03-formats/csv]
ReadSchema: struct<PULocationID:int,total_amount:double,tpep_pickup_datetime:timestamp_ntz>




## Type safety — JSON inference

In [8]:
infer_json = spark.read.json(str(out / 'json'))
infer_json.printSchema()
strict_json = spark.read.schema(sample.schema).json(str(out / 'json'))
strict_json.printSchema()

root
 |-- PULocationID: long (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- tpep_pickup_datetime: string (nullable = true)
 |-- VendorID: integer (nullable = true)

root
 |-- PULocationID: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- VendorID: integer (nullable = true)



## Iceberg deep dive (single table)

Table `local.default.workshop_iceberg` — **DataSource V2**, snapshot isolation, hidden partitioning, metadata tree under `data/output/warehouse/`.

In [6]:
def iceberg_storage_stats(short: str) -> dict:
    root = config.WAREHOUSE_DIR / 'default' / short
    if not root.exists():
        return {'data_files': 0, 'data_mb': 0.0, 'metadata_mb': 0.0, 'total_mb': 0.0}
    data = list((root / 'data').rglob('*.parquet')) if (root / 'data').exists() else []
    meta = list((root / 'metadata').rglob('*')) if (root / 'metadata').exists() else []
    data_b = sum(p.stat().st_size for p in data)
    meta_b = sum(p.stat().st_size for p in meta if p.is_file())
    return {
        'data_files': len(data),
        'data_mb': data_b / 1e6,
        'metadata_mb': meta_b / 1e6,
        'total_mb': (data_b + meta_b) / 1e6,
    }

def print_storage(label: str, stats: dict) -> None:
    print(
        f"{label}: {stats['data_files']} data files, "
        f"data {stats['data_mb']:.2f} MB, metadata {stats['metadata_mb']:.2f} MB, "
        f"total {stats['total_mb']:.2f} MB"
    )

def snapshot_ids(limit: int = 20):
    return spark.sql(
        f'SELECT snapshot_id, parent_id, committed_at FROM {ICEBERG}.snapshots '
        'ORDER BY committed_at'
    ).limit(limit).collect()

### Step 1 — initial load (partitioned table)

In [7]:
drop_iceberg_table(spark, ICEBERG)

(
    sample.writeTo(ICEBERG)
    .using('iceberg')
    .partitionedBy('PULocationID')
    .create()
)

stats_initial = iceberg_storage_stats(SHORT)
print_storage('After initial create', stats_initial)
spark.sql(f'SELECT snapshot_id, summary FROM {ICEBERG}.snapshots').show(truncate=False)
snap_create = snapshot_ids()[-1].snapshot_id
rows_create = spark.table(ICEBERG).count()
print(f'snapshot {snap_create} row count: {rows_create:,}')

After initial create: 261 data files, data 35.92 MB, metadata 0.03 MB, total 35.96 MB
+-------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|snapshot_id        |summary                                                                                                                                                                                                                                                                           

### Step 2 — mutate data (new snapshots)

Append rows, then **delete** — each commit adds a snapshot you can time-travel to.

In [8]:
extra = sample.limit(25_000).repartition(8)
extra.writeTo(ICEBERG).using('iceberg').append()
snap_after_append = snapshot_ids()[-1].snapshot_id
print_storage('After append', iceberg_storage_stats(SHORT))

spark.sql(f'''
    DELETE FROM {ICEBERG}
    WHERE VendorID = 2 AND total_amount > 75
''')
snap_after_delete = snapshot_ids()[-1].snapshot_id
print_storage('After delete', iceberg_storage_stats(SHORT))
spark.sql(f'SELECT snapshot_id, committed_at FROM {ICEBERG}.snapshots ORDER BY committed_at').show()

After append: 384 data files, data 36.28 MB, metadata 0.05 MB, total 36.33 MB


After delete: 624 data files, data 70.50 MB, metadata 0.12 MB, total 70.62 MB
+-------------------+--------------------+
|        snapshot_id|        committed_at|
+-------------------+--------------------+
|3948566375266761827|2026-11-24 10:02:...|
|1408966545729261547|2026-11-24 10:03:...|
|6334031544900861830|2026-11-24 10:03:...|
+-------------------+--------------------+



### Step 3 — time travel

In [12]:
def count_at_snapshot(snap_id: int) -> int:
    return spark.sql(f'SELECT COUNT(*) AS n FROM {ICEBERG} VERSION AS OF {snap_id}').collect()[0].n

for label, sid in [
    ('create', snap_create),
    ('after append', snap_after_append),
    ('after delete', snap_after_delete),
    ('current', None),
]:
    if sid is None:
        n = spark.table(ICEBERG).count()
    else:
        n = count_at_snapshot(sid)
    print(f'{label}: snapshot={sid} rows={n:,}')

# DataFrame reader variant
spark.read.option('versionAsOf', str(snap_create)).table(ICEBERG).count()

create: snapshot=7159666290121589577 rows=5,972,150
after append: snapshot=5356874784062155085 rows=5,997,150
after delete: snapshot=531403419228590680 rows=5,723,635
current: snapshot=None rows=5,723,635


5972150

### Step 4 — partition evolution

Add a **new hidden partition field** — metadata only; old files keep the old spec, new writes use the new spec.

In [9]:
spark.sql(f'DESCRIBE TABLE EXTENDED {ICEBERG}').filter(
    "col_name LIKE '%Partition%' or col_name = 'Provider'"
).show(truncate=False)
spark.sql(f'SELECT * FROM {ICEBERG}.partitions').show(10, truncate=False)

+-----------------------+---------+-------+
|col_name               |data_type|comment|
+-----------------------+---------+-------+
|# Partition Information|         |       |
|Provider               |iceberg  |       |
+-----------------------+---------+-------+

+---------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+----------------------+------------------------+
|partition|spec_id|record_count|file_count|total_data_file_size_in_bytes|position_delete_record_count|position_delete_file_count|equality_delete_record_count|equality_delete_file_count|last_updated_at       |last_updated_snapshot_id|
+---------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+----------------------+------------------------+
|{232}    |0      |7738        |2

In [10]:
spark.sql(f'ALTER TABLE {ICEBERG} ADD PARTITION FIELD bucket(16, VendorID)')
print('Added bucket(16, VendorID) — new appends land in evolved layout')

evo_batch = sample.limit(30_000).repartition(16, 'VendorID')
evo_batch.writeTo(ICEBERG).using('iceberg').append()

print_storage('After partition evolution + append', iceberg_storage_stats(SHORT))
spark.sql(f'SELECT * FROM {ICEBERG}.partitions').show(15, truncate=False)

Added bucket(16, VendorID) — new appends land in evolved layout


After partition evolution + append: 808 data files, data 70.96 MB, metadata 0.18 MB, total 71.14 MB
+---------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|partition|spec_id|record_count|file_count|total_data_file_size_in_bytes|position_delete_record_count|position_delete_file_count|equality_delete_record_count|equality_delete_file_count|last_updated_at        |last_updated_snapshot_id|
+---------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|{179}    |0      |1141        |1         |11539                        |0                           |0                         |0                           |0                         |2026-11-24

### Step 5 — repeated small writes (file sprawl)

Many tiny commits → more data files + metadata; compare volume before/after compaction.

In [11]:
tiny = sample.limit(3_000)
for i in range(6):
    tiny.repartition(4).writeTo(ICEBERG).using('iceberg').append()
stats_before_compact = iceberg_storage_stats(SHORT)
print_storage('After repeated small appends', stats_before_compact)

After repeated small appends: 1308 data files, data 71.82 MB, metadata 0.34 MB, total 72.16 MB


In [12]:
spark.sql('''
    CALL local.system.rewrite_data_files(
        table => 'default.workshop_iceberg',
        strategy => 'binpack',
        options => map(
            'rewrite-all', 'true',
            'min-input-files', '2'
        )
    )
''').show(truncate=False)

[Stage 267:============================================>        (136 + 6) / 162]

+--------------------------+----------------------+---------------------+-----------------------+--------------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|removed_delete_files_count|
+--------------------------+----------------------+---------------------+-----------------------+--------------------------+
|1004                      |445                   |35685960             |0                      |0                         |
+--------------------------+----------------------+---------------------+-----------------------+--------------------------+



In [17]:
stats_final = iceberg_storage_stats(SHORT)
print_storage('After rewrite_data_files', stats_final)
print('\nSummary')
for phase, st in [
    ('1 initial', stats_initial),
    ('2 after sprawl', stats_before_compact),
    ('3 after compaction', stats_final),
        ]:
    print_storage(phase, st)
wh = config.WAREHOUSE_DIR / 'default' / SHORT
print('warehouse path:', wh)
print('sample files:')
for p in sorted(wh.rglob('*'))[:25]:
    if p.is_file():
        print(f"  {p.relative_to(wh)}  ({p.stat().st_size / 1e3:.1f} KB)")

After rewrite_data_files: 1733 data files, data 106.62 MB, metadata 0.51 MB, total 107.13 MB

Summary
1 initial: 261 data files, data 35.89 MB, metadata 0.03 MB, total 35.92 MB
2 after sprawl: 1303 data files, data 71.79 MB, metadata 0.34 MB, total 72.13 MB
3 after compaction: 1733 data files, data 106.62 MB, metadata 0.51 MB, total 107.13 MB
warehouse path: /workspace/data/output/warehouse/default/workshop_iceberg
sample files:
  data/PULocationID=1/.00000-178-27292301-a35a-4890-8d82-0a807b5d8538-0-00131.parquet.crc  (0.0 KB)
  data/PULocationID=1/.00002-140-39a3ea4d-9ae0-46e9-a242-0d372b5f27ff-0-00008.parquet.crc  (0.1 KB)
  data/PULocationID=1/.00002-348-22d64584-da23-4283-87a8-11f63e138a70-0-00001.parquet.crc  (0.0 KB)
  data/PULocationID=1/00000-178-27292301-a35a-4890-8d82-0a807b5d8538-0-00131.parquet  (1.5 KB)
  data/PULocationID=1/00002-140-39a3ea4d-9ae0-46e9-a242-0d372b5f27ff-0-00008.parquet  (6.4 KB)
  data/PULocationID=1/00002-348-22d64584-da23-4283-87a8-11f63e138a70-0-00001.

In [13]:
# Current snapshot: files Iceberg would actually read
spark.sql(f"""
  SELECT COUNT(*) AS n_files,
         SUM(file_size_in_bytes) / 1e6 AS data_mb
  FROM {ICEBERG}.files
""").show()

# Everything under the table directory (includes old generations + metadata)
wh = config.WAREHOUSE_DIR / 'default' / SHORT  # adjust if your db/table path differs
data_bytes = sum(p.stat().st_size for p in wh.rglob('*.parquet') if p.is_file())
meta_bytes = sum(p.stat().st_size for p in wh.rglob('metadata/*') if p.is_file())
print(f"on-disk parquet: {data_bytes/1e6:.2f} MB  metadata: {meta_bytes/1e6:.2f} MB")

+-------+---------+
|n_files|  data_mb|
+-------+---------+
|    445|34.852577|
+-------+---------+

on-disk parquet: 106.67 MB  metadata: 0.51 MB


In [19]:
spark.sql(f"""
  SELECT snapshot_id, committed_at,
         summary['added-data-files'] AS added,
         summary['deleted-data-files'] AS deleted
  FROM {ICEBERG}.snapshots
  ORDER BY committed_at
""").show(truncate=False)

spark.sql(f"SELECT COUNT(*) AS refs FROM {ICEBERG}.refs").show()

+-------------------+-----------------------+-----+-------+
|snapshot_id        |committed_at           |added|deleted|
+-------------------+-----------------------+-----+-------+
|7159666290121589577|2026-11-17 12:38:57.261|261  |NULL   |
|5356874784062155085|2026-11-17 12:38:58.888|165  |NULL   |
|531403419228590680 |2026-11-17 12:39:01.919|240  |324    |
|856533675022128632 |2026-11-17 12:39:03.809|151  |NULL   |
|4929839525658207239|2026-11-17 12:39:04.549|76   |NULL   |
|812362283617942518 |2026-11-17 12:39:05.101|101  |NULL   |
|7183157620275531373|2026-11-17 12:39:05.646|76   |NULL   |
|2181034755183755590|2026-11-17 12:39:06.14 |85   |NULL   |
|173997204236967436 |2026-11-17 12:39:06.676|74   |NULL   |
|6803620034453484451|2026-11-17 12:39:07.129|74   |NULL   |
|7851274452505826608|2026-11-17 12:39:12.02 |430  |979    |
+-------------------+-----------------------+-----+-------+

+----+
|refs|
+----+
|   1|
+----+



In [20]:
DB_TABLE = ICEBERG.split('.', 1)[1]

spark.sql(f"""
  CALL local.system.expire_snapshots(
    table => '{DB_TABLE}',
    older_than => current_timestamp(),
    retain_last => 1
  )
""").show(truncate=False)

+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|deleted_data_files_count|deleted_position_delete_files_count|deleted_equality_delete_files_count|deleted_manifest_files_count|deleted_manifest_lists_count|deleted_statistics_files_count|
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|1303                    |0                                  |0                                  |13                          |10                          |0                             |
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+



In [21]:
spark.sql(f"""
  SELECT snapshot_id, committed_at
  FROM {ICEBERG}.snapshots
  ORDER BY committed_at
""").show()

+-------------------+--------------------+
|        snapshot_id|        committed_at|
+-------------------+--------------------+
|7851274452505826608|2026-11-17 12:39:...|
+-------------------+--------------------+



# Implication of storage format in query plans

## Read-side

In [5]:
from pyspark.sql import functions as F
from shared.utils import timed_action

VID = 2

def q(df):
    return (
        df.agg(F.count("*").alias("n"), F.sum("total_amount").alias("s"))
    )

readers = {
    "csv": spark.read.option("header", True).schema(sample.schema)
          .csv(f"{out}/csv/VendorID={VID}"),
    "json": spark.read.schema(sample.schema).json(f"{out}/json/VendorID={VID}"),
    "parquet": spark.read.parquet(f"{out}/parquet/VendorID={VID}"),
    "iceberg": spark.table("local.default.io_lab_sample").filter(F.col("VendorID") == VID),
}

results = []
for fmt, raw in readers.items():
    query = q(raw)
    print("\n" + "=" * 60)
    print(f"READ: {fmt}  (VendorID={VID})")
    print("=" * 60)
    query.explain("formatted")
    spark.catalog.clearCache()
    n, secs = timed_action(query, action="count")
    row = query.collect()[0]
    print(f"time: {secs:.2f}s  agg rows: {n}  n={row['n']:,}  sum={row['s']}")
    results.append((fmt, secs, row["n"], row["s"]))

print("\n--- summary ---")
for fmt, secs, n, s in results:
    print(f"{fmt:8}  {secs:6.2f}s  count={n:,}  sum={s}")


READ: csv  (VendorID=2)
== Physical Plan ==
AdaptiveSparkPlan (5)
+- HashAggregate (4)
   +- Exchange (3)
      +- HashAggregate (2)
         +- Scan csv  (1)


(1) Scan csv 
Output [1]: [total_amount#95]
Batched: false
Location: InMemoryFileIndex [file:/workspace/data/output/03-formats/csv/VendorID=2]
ReadSchema: struct<total_amount:double>

(2) HashAggregate
Input [1]: [total_amount#95]
Keys: []
Functions [2]: [partial_count(1), partial_sum(total_amount#95)]
Aggregate Attributes [2]: [count#122L, sum#123]
Results [2]: [count#124L, sum#125]

(3) Exchange
Input [2]: [count#124L, sum#125]
Arguments: SinglePartition, ENSURE_REQUIREMENTS, [plan_id=362]

(4) HashAggregate
Input [2]: [count#124L, sum#125]
Keys: []
Functions [2]: [count(1), sum(total_amount#95)]
Aggregate Attributes [2]: [count(1)#119L, sum(total_amount#95)#120]
Results [2]: [count(1)#119L AS n#113L, sum(total_amount#95)#120 AS s#114]

(5) AdaptiveSparkPlan
Output [2]: [n#113L, s#114]
Arguments: isFinalPlan=false




time: 0.21s  agg rows: 1  n=4,506,016  sum=None

READ: json  (VendorID=2)
== Physical Plan ==
AdaptiveSparkPlan (5)
+- HashAggregate (4)
   +- Exchange (3)
      +- HashAggregate (2)
         +- Scan json  (1)


(1) Scan json 
Output [1]: [total_amount#99]
Batched: false
Location: InMemoryFileIndex [file:/workspace/data/output/03-formats/json/VendorID=2]
ReadSchema: struct<total_amount:double>

(2) HashAggregate
Input [1]: [total_amount#99]
Keys: []
Functions [2]: [partial_count(1), partial_sum(total_amount#99)]
Aggregate Attributes [2]: [count#144L, sum#145]
Results [2]: [count#146L, sum#147]

(3) Exchange
Input [2]: [count#146L, sum#147]
Arguments: SinglePartition, ENSURE_REQUIREMENTS, [plan_id=444]

(4) HashAggregate
Input [2]: [count#146L, sum#147]
Keys: []
Functions [2]: [count(1), sum(total_amount#99)]
Aggregate Attributes [2]: [count(1)#141L, sum(total_amount#99)#142]
Results [2]: [count(1)#141L AS n#135L, sum(total_amount#99)#142 AS s#136]

(5) AdaptiveSparkPlan
Output [2]: [n#

## Write-side

In [8]:
out = config.OUTPUT_BASE / "03-formats"
sample.createOrReplaceTempView("write_sample")

def explain_write_sql(sql: str, label: str) -> None:
    print("\n" + "=" * 60)
    print(f"WRITE PLAN: {label}")
    print("=" * 60)
    row = spark.sql(sql).collect()[0]
    plan = row["plan"] if "plan" in row.asDict() else row[0]
    print(plan)

explain_write_sql(f"""
EXPLAIN FORMATTED
INSERT OVERWRITE DIRECTORY '{out / "parquet-write-plan"}'
USING parquet
SELECT * FROM write_sample
""", "parquet")

explain_write_sql(f"""
EXPLAIN FORMATTED
INSERT OVERWRITE DIRECTORY '{out / "json-write-plan"}'
USING json
SELECT * FROM write_sample
""", "json")

explain_write_sql(f"""
EXPLAIN FORMATTED
INSERT OVERWRITE DIRECTORY '{out / "csv-write-plan"}'
USING csv
OPTIONS (header 'true')
SELECT * FROM write_sample
""", "csv")

explain_write_sql(f"""
EXPLAIN FORMATTED
INSERT INTO {ICEBERG}
SELECT * FROM write_sample
""", "WRITE PLAN: iceberg (INSERT)")


WRITE PLAN: parquet
== Physical Plan ==
Execute InsertIntoDataSourceDirCommand (1)
   +- InsertIntoDataSourceDirCommand (2)
         +- Project (7)
            +- SubqueryAlias (6)
               +- View (5)
                  +- Project (4)
                     +- LogicalRelation (3)


(1) Execute InsertIntoDataSourceDirCommand
Output: []

(2) InsertIntoDataSourceDirCommand
Arguments: Storage(Location: /workspace/data/output/03-formats/parquet-write-plan), parquet, true

(3) LogicalRelation
Arguments: parquet, [VendorID#0, tpep_pickup_datetime#1, tpep_dropoff_datetime#2, passenger_count#3L, trip_distance#4, RatecodeID#5L, store_and_fwd_flag#6, PULocationID#7, DOLocationID#8, payment_type#9L, fare_amount#10, extra#11, mta_tax#12, tip_amount#13, tolls_amount#14, improvement_surcharge#15, total_amount#16, congestion_surcharge#17, Airport_fee#18], false

(4) Project
Arguments: [VendorID#0, PULocationID#7, total_amount#16, tpep_pickup_datetime#1]

(5) View
Arguments: `write_sample`, true, 